In [1]:
import sys, time
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
sys.path.insert(0, '/tmp/fwigan-whole-workflow')
import notebook_workflow_comparison as wf
WF_OUTPUT = Path('/tmp/fwigan-whole-workflow/workflow')
WF_NAME = 'FWIGAN_Noisy'
WF_DEADLINE = 1791892403
WF_REPO = Path('/tmp/fwigan-whole-workflow/repo')


In [2]:
from pathlib import Path
import torch
from fwigan.cli import load_config, prepare_data
from fwigan.experiment import Experiment
from fwigan.plotting import plot_results

ROOT = Path.cwd()
if not (ROOT / "configs").is_dir():
    ROOT = ROOT.parent
CONFIG = Path('/tmp/fwigan-whole-workflow/workflow/new-noisy.yaml')
workflow = load_config(CONFIG)
from fwigan.data import load_observations
data = load_observations('/tmp/fwigan-whole-workflow/workflow/shared_observations.pt')
run = Experiment.create(workflow.experiment, data, WF_OUTPUT / "FWIGAN_Noisy-new-runs")
initial_metrics = run.evaluate()
print(initial_metrics)
wf.new_epoch(run, 0, WF_OUTPUT, WF_NAME, WF_DEADLINE)


/tmp/fwigan-whole-workflow/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'epoch': 0, 'batch': 0, 'global_step': 0, 'solver_forward': 1, 'solver_adjoint': 0, 'raw_residual_mse': 13.345551490783691, 'model_rmse': 478.00250244140625}


In [3]:
for epoch in range(1, min(2, workflow.experiment.epochs) + 1):
    wf.check_deadline(WF_DEADLINE)
    run.run(epochs=epoch)
    wf.new_epoch(run, epoch, WF_OUTPUT, WF_NAME, WF_DEADLINE)


In [4]:
split_id, split_fingerprint = run.run_id, run.data_fingerprint
del run
run = Experiment.resume(next((WF_OUTPUT / (WF_NAME + "-new-runs")).glob("fwigan-*/checkpoint.pt")), device=workflow.experiment.device)
assert run.run_id == split_id and run.data_fingerprint == split_fingerprint
for epoch in range(3, workflow.experiment.epochs + 1):
    wf.check_deadline(WF_DEADLINE)
    run.run(epochs=epoch)
    wf.new_epoch(run, epoch, WF_OUTPUT, WF_NAME, WF_DEADLINE)
result = torch.load(run.export_final(), map_location="cpu")
figures = plot_results(data, result, output_dir=WF_OUTPUT / (WF_NAME + "-new-figures"))
wf.close_figures()
